# Trankit zero-shot evaluation on the fixed Cantonese-HK split

This notebook evaluates off-the-shelf Trankit on the project's fixed UD Cantonese-HK r2.18 split. It uses **gold sentence boundaries and gold tokenisation** so LAS is comparable with the existing Stanza experiments. No Cantonese training or model adaptation is performed.

Two official Trankit XLM-R-large packages are compared on **dev only**: `chinese` (UD Simplified Chinese-GSDSimp) and `traditional-chinese` (UD Chinese-GSD). The higher-dev-LAS package is then evaluated once on the fixed 100-sentence test. The test was already used elsewhere in this project for model-family comparison, so it is not an untouched confirmatory test.

Select a GPU runtime (A100 recommended) and choose **Runtime → Run all**. No Google Drive or uploaded files are required.

In [ ]:
# Trankit 1.1.2 declares an obsolete torch upper bound. Install its pinned source
# without dependencies, then supply a tested adapters/transformers combination.
%pip install -q transformers==4.51.3 adapters==1.2.0 sentencepiece sacremoses protobuf langid==1.1.6
%pip install -q --no-deps git+https://github.com/nlp-uoregon/trankit.git@54e863327391262cf72f6adc1b0ff104e972a1dc

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, random, shutil, importlib.util, platform
import urllib.request
import numpy as np
import torch

SEED=42
WORK=Path('/content/yue_trankit_eval')
DATA=WORK/'data'; CACHE=WORK/'trankit_cache'; OUT=WORK/'outputs'
# Make Run all repeatable in the same runtime while retaining downloaded models.
for p in (DATA,OUT):
    if p.exists(): shutil.rmtree(p)
for p in (DATA,CACHE,OUT): p.mkdir(parents=True,exist_ok=True)

TRANKIT_COMMIT='54e863327391262cf72f6adc1b0ff104e972a1dc'
EMBEDDING='xlm-roberta-large'
LANGUAGES=['chinese','traditional-chinese']
PROJECT_COMMIT='cd6359d4967b27d7a65d4da4182ae2141d26bacc'
DATA_BASE=f'https://raw.githubusercontent.com/ss-sebastian/zh-yue-d2d/{PROJECT_COMMIT}/data/processed/yue_hk'
SPLIT_SHA256={'dev':'41bc28d903457e70a4747307e56b3eb7820f2e6d3ede10ab549fd4a85aef63b7',
              'test':'1d7b19ac4c0a75d58a80482413ff9cac63ed18917262c18fa1f92fc0f871c0a0'}
EVAL_URL='https://universaldependencies.org/conll18/conll18_ud_eval.py'
EVAL_SHA256='1072e02af00b1a56205b5e8216d51dee9b8944a104d80744afaccc78859fcb16'

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''): h.update(chunk)
    return h.hexdigest()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), 'Select a Colab GPU runtime before Run all.'
print('Python',platform.python_version(),'PyTorch',torch.__version__,'GPU',torch.cuda.get_device_name(0))

for split,expected in SPLIT_SHA256.items():
    path=DATA/f'{split}.conllu'
    urllib.request.urlretrieve(f'{DATA_BASE}/{split}.conllu',path)
    assert sha256(path)==expected,(split,sha256(path),expected)
    print(split,sha256(path))

eval_path=WORK/'conll18_ud_eval.py'
urllib.request.urlretrieve(EVAL_URL,eval_path)
assert sha256(eval_path)==EVAL_SHA256
spec=importlib.util.spec_from_file_location('official_conll18',eval_path)
official=importlib.util.module_from_spec(spec); spec.loader.exec_module(official)
def conll18(gold,system):
    return official.evaluate(official.load_conllu_file(str(gold)),official.load_conllu_file(str(system)))

In [ ]:
# Compatibility shims for the pinned legacy Trankit source on current Colab.
import transformers
if not hasattr(transformers,'AdamW'):
    transformers.AdamW=torch.optim.AdamW
_torch_load=torch.load
def _legacy_torch_load(*args,**kwargs):
    kwargs.setdefault('weights_only',False)
    return _torch_load(*args,**kwargs)
torch.load=_legacy_torch_load

import adapters, trankit
from trankit import Pipeline
assert trankit.__version__=='1.1.2',trankit.__version__
assert all(lang in trankit.supported_langs for lang in LANGUAGES)
print('Trankit',trankit.__version__,'Transformers',transformers.__version__,'Adapters',adapters.__version__)

pipeline=Pipeline(LANGUAGES[0],gpu=True,cache_dir=str(CACHE),embedding=EMBEDDING)
pipeline.add(LANGUAGES[1])
print('Loaded:',pipeline.added_langs,'embedding:',EMBEDDING)

In [ ]:
def split_blocks(text): return text.strip().split('\n\n')
def integer_rows(block):
    return [line.split('\t') for line in block.splitlines()
            if line and not line.startswith('#') and line.split('\t',1)[0].isdigit()]

def predict_pretokenized(language,gold_path,output_path):
    pipeline.set_active(language)
    blocks=split_blocks(Path(gold_path).read_text(encoding='utf-8'))
    forms=[[row[1] for row in integer_rows(block)] for block in blocks]
    result=pipeline.posdep(forms)
    predicted=result['sentences']
    assert len(predicted)==len(blocks)
    output=[]
    for block,pred_sent,gold_forms in zip(blocks,predicted,forms):
        pred_tokens=pred_sent['tokens']
        assert len(pred_tokens)==len(gold_forms),(len(pred_tokens),len(gold_forms),gold_forms)
        pred_iter=iter(pred_tokens); lines=[]
        for line in block.splitlines():
            if line and not line.startswith('#') and line.split('\t',1)[0].isdigit():
                cols=line.split('\t'); token=next(pred_iter)
                assert int(cols[0])==int(token['id']) and cols[1]==token['text'],(cols[:2],token)
                cols[3]=token.get('upos','_') or '_'
                cols[4]=token.get('xpos','_') or '_'
                cols[5]=token.get('feats','_') or '_'
                cols[6]=str(token['head'])
                cols[7]=token['deprel']
                line='\t'.join(cols)
            lines.append(line)
        try: next(pred_iter); raise AssertionError('Extra Trankit token')
        except StopIteration: pass
        output.append('\n'.join(lines))
    output_path.write_text('\n\n'.join(output)+'\n\n',encoding='utf-8')
    assert len(split_blocks(output_path.read_text(encoding='utf-8')))==len(blocks)
    return output_path

def strict_las(gold_path,pred_path,exclude_punct=False):
    gold=[r for b in split_blocks(Path(gold_path).read_text(encoding='utf-8')) for r in integer_rows(b)]
    pred=[r for b in split_blocks(Path(pred_path).read_text(encoding='utf-8')) for r in integer_rows(b)]
    assert len(gold)==len(pred) and all(g[:2]==p[:2] for g,p in zip(gold,pred))
    pairs=[(g,p) for g,p in zip(gold,pred) if not(exclude_punct and g[3]=='PUNCT')]
    return sum(g[6]==p[6] and g[7]==p[7] for g,p in pairs)/len(pairs)

def score_record(gold,pred):
    score=conll18(gold,pred)
    return {'sentences':len(split_blocks(Path(gold).read_text(encoding='utf-8'))),
            'tokens':sum(len(integer_rows(b)) for b in split_blocks(Path(gold).read_text(encoding='utf-8'))),
            'UPOS_percent':100*score['UPOS'].f1,'UAS_percent':100*score['UAS'].f1,
            'CoNLL18_LAS_percent':100*score['LAS'].f1,
            'strict_LAS_percent':100*strict_las(gold,pred),
            'strict_LAS_no_punct_percent':100*strict_las(gold,pred,True),
            'prediction_sha256':sha256(pred)}

In [ ]:
# Compare language packages on DEV only.
dev_results={}
for language in LANGUAGES:
    pred=OUT/f'dev.{language}.pred.conllu'
    predict_pretokenized(language,DATA/'dev.conllu',pred)
    dev_results[language]=score_record(DATA/'dev.conllu',pred)
    print(language,json.dumps(dev_results[language],ensure_ascii=False,indent=2))

# Deterministic tie policy: prefer traditional-chinese only when LAS is exactly tied.
selected=max(LANGUAGES,key=lambda x:(dev_results[x]['CoNLL18_LAS_percent'],x=='traditional-chinese'))
print('DEV-selected Trankit package:',selected)

## Fixed test — evaluate the dev-selected package once

The cell below does not compare both packages on test. It evaluates only the package selected by dev LAS.

In [ ]:
sentinel=OUT/'.test_completed'
assert not sentinel.exists(),'Test already evaluated in this runtime directory.'
test_pred=OUT/f'test.{selected}.pred.conllu'
predict_pretokenized(selected,DATA/'test.conllu',test_pred)
test_result=score_record(DATA/'test.conllu',test_pred)
sentinel.write_text(json.dumps({'selected':selected,'prediction_sha256':sha256(test_pred)}),encoding='utf-8')

# Record every downloaded Trankit/XLM-R file by path, size and SHA-256.
model_files=[]
for path in sorted(p for p in CACHE.rglob('*') if p.is_file()):
    model_files.append({'path':str(path.relative_to(CACHE)),'bytes':path.stat().st_size,'sha256':sha256(path)})

result={'experiment':'off-the-shelf Trankit zero-shot Cantonese-HK dependency parsing',
        'condition':{'gold_sentence_boundaries':True,'gold_tokenization':True,
                     'predicted_pos_morphology':True,'cantonese_finetuning':False},
        'selection':'Compare Chinese packages on dev CoNLL-2018 LAS; test selected package once',
        'selected_package':selected,'dev_results':dev_results,'test_result':test_result,
        'software':{'trankit_version':trankit.__version__,'trankit_commit':TRANKIT_COMMIT,
                    'embedding':EMBEDDING,'transformers':transformers.__version__,
                    'adapters':adapters.__version__,'torch':torch.__version__},
        'data':{'project_commit':PROJECT_COMMIT,'split_sha256':SPLIT_SHA256},
        'model_file_manifest':model_files,
        'test_selection_caveat':'This project test was previously used for model-family comparison and is not untouched.'}
(OUT/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps({k:v for k,v in result.items() if k!='model_file_manifest'},ensure_ascii=False,indent=2))

In [ ]:
# Package only results and predictions, not the multi-gigabyte model cache.
bundle=WORK/'download_bundle'; bundle.mkdir(exist_ok=True)
for path in [OUT/'result.json',test_pred]+[OUT/f'dev.{lang}.pred.conllu' for lang in LANGUAGES]:
    shutil.copy2(path,bundle/path.name)
archive=Path(shutil.make_archive('/content/yue_trankit_results','zip',root_dir=bundle))
print('archive:',archive,'SHA-256:',sha256(archive))
from google.colab import files
files.download(str(archive))